# 視覚的注意機構（Visual Attention）による画像キャプション生成 

## 学習目標
1. 画像キャプション生成モデルの作成方法を学ぶ
2. テキスト生成モデルのトレーニングと予測方法を学ぶ

画像キャプション生成モデルは画像を入力として受け取り、テキストを出力します。理想的には、人間が付けるキャプションと同様に、画像内の出来事や対象物を正確に説明するテキストをモデルが出力することが望まれます。<br>
たとえば、以下の例のような画像が与えられた場合、モデルは *"some people are playing baseball."*（何人かの人々が野球をしている）といったキャプションを生成することが期待されます。

<div><img src="../sample_images/baseball.jpeg" width="500"></div>

テキストを生成するために、エンコーダー・デコーダーモデルを構築します。エンコーダーは入力画像の埋め込み（embedding）を出力し、デコーダーはその画像埋め込みからテキストを出力します。<br>

このノートブックでは、[Show, Attend and Tell: Neural Image Caption Generation with Visual Attention](https://arxiv.org/abs/1502.03044) と同様のモデルアーキテクチャを使用し、Attention ベースの画像キャプション生成モデルを構築します。

このノートブックはエンドツーエンドの例です。トレーニングデータセットには、COCO 大規模物体検出・セグメンテーション・キャプションデータセットを使用します。

## 準備

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import time
from textwrap import wrap

import keras
import matplotlib.pylab as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from keras import Input
from keras.layers import (
    GRU,
    Add,
    Attention,
    Concatenate,
    Dense,
    Embedding,
    LayerNormalization,
    Rescaling,
    Reshape,
    Resizing,
    StringLookup,
    TextVectorization,
)

print(keras.version())

## データセットの読み込みと準備

TensorFlow Datasets の機能を使用して、[COCO captions](https://www.tensorflow.org/datasets/catalog/coco_captions) データセットを読み込みます。
このバージョンには、COCO 2014 の画像、バウンディングボックス、ラベル、およびキャプションが含まれており、Karpathy and Li (2015) によって定義されたサブセットに分割され、元のデータセットのいくつかのデータ品質問題（たとえば、元のデータセットの一部の画像にキャプションがなかった問題など）が処理されています。

まず、定数をいくつか定義しましょう。<br>
このラボでは、特徴量抽出器（feature extractor）として `keras.applications` の事前トレーニング済み [InceptionResNetV2](https://keras.io/api/applications/inceptionresnetv2/) モデルを使用するため、いくつかの定数は InceptionResNetV2 モデルの定義に基づいています。<br>
したがって、他の種類のベースモデルを使用する場合は、これらの定数も必ず変更してください。

`keras.applications` は、一般的で安定した画像モデルを提供する事前トレーニング済みモデルのリポジトリです。<br>
一部のプラットフォームではコンパイルされたモデルグラフとパラメータのみを含むモデルが提供されるのに対し、`keras.applications` はモデルのメタデータを含んでおり、モデルの動作にアクセスして制御できるため柔軟性があります。<br>
そのため、たとえばレイヤーのメタデータにアクセスすることで、モデルの最終レイヤーからの出力（例: CNN モデルの平坦化された 1D Tensor 出力）だけでなく、中間レイヤーからの出力（例: 中間 3D Tensor）も取得できます。

In [ ]:
# 精度/速度を制御するためにこれらを変更します
VOCAB_SIZE = 20000  # 収束を早めるためにより少ない単語数を使用
ATTENTION_DIM = 512  # Attention の Dense レイヤーのサイズ
WORD_EMBEDDING_DIM = 128

# InceptionResNetV2 は (299, 299, 3) の画像を入力として受け取り、
# (8, 8, 1536) 形状の特徴量を返します
FEATURE_EXTRACTOR = keras.applications.inception_resnet_v2.InceptionResNetV2(
    include_top=False, weights="imagenet"
)
IMG_HEIGHT = 299
IMG_WIDTH = 299
IMG_CHANNELS = 3
FEATURES_SHAPE = (8, 8, 1536)

resize = Resizing(IMG_HEIGHT, IMG_WIDTH)
rescale = Rescaling(1.0 / 255)

### フィルタリングと前処理
ここではデータセットの前処理を行います。以下の関数は次の処理を実行します:
- 画像のサイズを (`IMG_HEIGHT`, `IMG_WIDTH`) の形状にリサイズ
- ピクセル値を [0, 255] から [0, 1] にリスケール
- 画像（`image_tensor`）とキャプション（`caption`）の辞書を返す

**注**: このデータセットはローカル環境に保存するには大きすぎます。そのため、us-central1 にある公開 GCS バケットに保存されています。
米国以外の場所にあるノートブックからアクセスする場合、(a) 処理が遅くなり、(b) ネットワーク料金が発生する可能性があります。

In [ ]:
GCS_DIR = "gs://asl-public-data/data/tensorflow_datasets/"
BUFFER_SIZE = 1000


def get_image_label(example):
    caption = example["captions"]["text"][0]  # 画像ごとに最初のキャプションのみ使用
    img = example["image"]
    img = resize(img)
    img = rescale(img)
    return {"image_tensor": img, "caption": caption}


trainds = tfds.load("coco_captions", split="train", data_dir=GCS_DIR)

trainds = trainds.map(
    get_image_label, num_parallel_calls=tf.data.AUTOTUNE
).shuffle(BUFFER_SIZE)
trainds = trainds.prefetch(buffer_size=tf.data.AUTOTUNE)

### データの可視化
データセット内の画像とサンプルキャプションを見てみましょう。

In [ ]:
f, ax = plt.subplots(1, 4, figsize=(20, 5))
for idx, data in enumerate(trainds.take(4)):
    ax[idx].imshow(data["image_tensor"].numpy())
    caption = "\n".join(wrap(data["caption"].numpy().decode("utf-8"), 30))
    ax[idx].set_title(caption)
    ax[idx].axis("off")

## テキストの前処理

文の開始（`<start>`）と終了（`<end>`）を表す特別なトークンを追加します。<br>
ここではエンコーダー・デコーダーモデルを使用しており、予測時にキャプション生成を開始するために `<start>` を使用し、キャプションの長さが可変であるため `<end>` トークンが確認された時点で予測を終了させることから、開始トークンと終了トークンを追加します。

次に、さらなる前処理のためにキャプションの完全なリストを作成します。

In [ ]:
def add_start_end_token(data):
    start = keras.ops.convert_to_tensor("<start>")
    end = keras.ops.convert_to_tensor("<end>")
    data["caption"] = tf.strings.join(
        [start, data["caption"], end], separator=" "
    )
    return data


trainds = trainds.map(add_start_end_token)

## キャプションの前処理とトークン化

次の手順に従って、[TextVectorization](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) レイヤーを使用してテキストキャプションを整数シーケンスに変換します:

* [adapt](https://github.com/keras-team/keras/blob/adbfd13426a0da9864d9a0fcd5be5eed74ca341f/keras/src/layers/preprocessing/text_vectorization.py#L387) を使用してすべてのキャプションを反復処理し、キャプションを単語に分割して、上位 `VOCAB_SIZE` 単語の語彙を計算します。
* 各単語を語彙内のインデックスにマッピングして、すべてのキャプションをトークン化します。すべての出力シーケンスは `MAX_CAPTION_LEN` の長さにパディングされます。ここではこのデータセットに十分な値として `64` を直接指定していますが、データセット内の非常に長い文を切り捨てたくない場合は、データセット全体を処理してこの値を計算する必要があることに注意してください。

**注**: この処理には約 10 分かかります。

In [ ]:
MAX_CAPTION_LEN = 64


# <start> と <end> のトークンを保持するために、"<>" 文字を保持するよう
# TextVectorization のデフォルトの標準化処理をオーバーライドします。
def standardize(inputs):
    inputs = tf.strings.lower(inputs)
    return tf.strings.regex_replace(
        inputs, r"[!\"#$%&\(\)\*\+.,-/:;=?@\[\\\]^_`{|}~]?", ""
    )


# 語彙から最も頻出する単語を選択し、句読点などを削除します。
tokenizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    standardize=standardize,
    output_sequence_length=MAX_CAPTION_LEN,
)

tokenizer.adapt(trainds.map(lambda x: x["caption"]))

サンプルテキストをトークン化してみましょう

In [ ]:
tokenizer(["<start> This is a sentence <end>"])

In [ ]:
sample_captions = []
for d in trainds.take(5):
    sample_captions.append(d["caption"].numpy())

In [ ]:
sample_captions

In [ ]:
print(tokenizer(sample_captions))

すべての文が同じトークン（例: '3' と '4'）で始まり、終わっていることに注目してください。これらの値はそれぞれ開始トークンと終了トークンを表しています。

ID を元のテキストに戻すこともできます。

In [ ]:
for wordid in tokenizer([sample_captions[0]])[0]:
    print(tokenizer.get_vocabulary()[wordid], end=" ")

また、`StringLookup` レイヤーを使用して単語とインデックスの相互変換ツール（Word <-> Index converter）を作成することもできます。

In [ ]:
# ルックアップテーブル: 単語 -> インデックス
word_to_index = StringLookup(
    mask_token="", vocabulary=tokenizer.get_vocabulary()
)

# ルックアップテーブル: インデックス -> 単語
index_to_word = StringLookup(
    mask_token="", vocabulary=tokenizer.get_vocabulary(), invert=True
)

### トレーニング用 tf.data データセットの作成
適応（adapt）させたトークン化をすべてのサンプルに適用し、トレーニング用の tf.data Dataset を作成しましょう。

ここでは、特徴量キャプションからテキストをシフトしてラベルを作成している点に注意してください。<br>
入力キャプションが `"<start> I love cats <end>"` の場合、そのラベルは `"I love cats <end> <padding>"` になります。<br>
これにより、モデルは `<start>` から `I` を予測することを学習できます。

データセットはタプルを返す必要があり、最初の要素が特徴量（`image_tensor` と `caption`）、2 番目の要素がラベル（ターゲット）です。

In [ ]:
BATCH_SIZE = 32


def create_ds_fn(data):
    img_tensor = data["image_tensor"]
    caption = tokenizer(data["caption"])

    target = keras.ops.roll(caption, -1, 0)
    zeros = keras.ops.zeros([1], dtype="int32")
    target = keras.ops.concatenate((target[:-1], zeros), axis=-1)
    return (img_tensor, caption), target


batched_ds = (
    trainds.map(create_ds_fn)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

いくつかの例を見てみましょう。

In [ ]:
for (img, caption), label in batched_ds.take(2):
    print(f"Image shape: {img.shape}")
    print(f"Caption shape: {caption.shape}")
    print(f"Label shape: {label.shape}")
    print(caption[0])
    print(label[0])

## モデル
それでは画像キャプション生成モデルを設計しましょう。<br>
これは画像エンコーダーと、それに続くキャプションデコーダーで構成されます。

### 画像エンコーダー
画像エンコーダーモデルは非常にシンプルです。事前トレーニング済みモデルを通じて特徴量を抽出し、全結合（Dense）レイヤーに渡します。

1. この例では、InceptionResNetV2 の畳み込みレイヤーから特徴量を抽出し、(Batch Size, 8, 8, 1536) のベクトルを取得します。
1. ベクトルを (Batch Size, 64, 1536) にリシェイプ（Reshape）します。
1. Dense レイヤーを使用して `ATTENTION_DIM` の長さに圧縮し、(Batch Size, 64, ATTENTION_DIM) を返します。
1. 後で Attention レイヤーが画像全体にアテンションを適用して次の単語を予測します。

In [ ]:
FEATURE_EXTRACTOR.trainable = False

image_input = Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
image_features = FEATURE_EXTRACTOR(image_input)

x = Reshape((FEATURES_SHAPE[0] * FEATURES_SHAPE[1], FEATURES_SHAPE[2]))(
    image_features
)
encoder_output = Dense(ATTENTION_DIM, activation="relu")(x)

In [ ]:
encoder = keras.Model(inputs=image_input, outputs=encoder_output)
encoder.summary()

### キャプションデコーダー
キャプションデコーダーには、入力画像のさまざまな部分に焦点を当てるアテンション機構が組み込まれています。

#### アテンションヘッド

デコーダーはアテンションを使用して、入力シーケンスの特定の部分を選択的に注視します。
アテンションは各サンプルの入力としてベクトルのシーケンスを受け取り、各サンプルに対して「アテンション（コンテキスト）」ベクトルを返します。

これがどのように機能するかを見てみましょう:

<img src="https://user-images.githubusercontent.com/6895245/173408554-d4b6387b-248b-421e-8911-550d0561d001.png" alt="attention equation 1" width="800">

<img src="https://user-images.githubusercontent.com/6895245/173408648-38c6b582-a68b-4697-982a-1d885b83dd0b.png" alt="attention equation 2" width="800">

ここで:

* $s$ はエンコーダーのインデックスです。
* $t$ はデコーダーのインデックスです。
* $\alpha_{ts}$ はアテンションの重みです。
* $h_s$ は注視されるエンコーダー出力のシーケンスです（Transformer の用語におけるアテンションの「key」と「value」）。
* $h_t$ はシーケンスを注視するデコーダーの状態です（Transformer の用語におけるアテンションの「query」）。
* $c_t$ は結果として得られるコンテキストベクトルです。

上記の数式:

1. エンコーダーの出力シーケンス全体に対するソフトマックスとして、アテンションの重み $\alpha_{ts}$ を計算します。
2. エンコーダー出力の重み付き合計としてコンテキストベクトルを計算します。

最後は $score$ 関数です。その役割は、各 key-query ペアのスカラーロジットスコアを計算することです。一般的なアプローチが 2 つあります:

<img src="https://user-images.githubusercontent.com/6895245/173408773-3781cacc-de00-49c6-9909-f6cd65a0501b.png" alt="attention equation 4" width="800">

このノートブックでは、事前定義された `layers.Attention` を使用して Luong スタイルのアテンションを実装します。

#### デコーダーのステップ

デコーダーの役割は、次の出力トークンの予測を生成することです。

1. デコーダーは現在の単語トークンをバッチとして受け取ります。
1. 単語トークンを `ATTENTION_DIM` 次元に埋め込み（Embed）ます。
1. GRU レイヤーが単語の埋め込みを追跡し、GRU の出力と状態を返します。
1. Luong スタイルのアテンションは、GRU 出力をクエリとして使用して、エンコーダーの出力特徴量全体にアテンションを適用します。
1. アテンション出力と GRU 出力が加算され（スキップ接続）、レイヤー正規化（Layer Normalization）レイヤーで正規化されます。
1. GRU 出力に基づいて次のトークンのロジット予測を生成します。

Keras Functional API でこれらすべてのステップを定義できますが、推論フェーズでレイヤーと重みを再利用できるように、ここではトレーニング可能なパラメータを持つレイヤーをインスタンス化していることに注意してください。

In [ ]:
word_input = Input(shape=(MAX_CAPTION_LEN,), name="words")
embed_x = Embedding(VOCAB_SIZE, ATTENTION_DIM)(word_input)

decoder_gru = GRU(
    ATTENTION_DIM,
    return_sequences=True,
    return_state=True,
)
gru_output, gru_state = decoder_gru(embed_x)

decoder_attention = Attention()
context_vector = decoder_attention([gru_output, encoder_output])

addition = Add()([gru_output, context_vector])

layer_norm = LayerNormalization(axis=-1)
layer_norm_out = layer_norm(addition)

decoder_output_dense = Dense(VOCAB_SIZE)
decoder_output = decoder_output_dense(layer_norm_out)

In [ ]:
decoder = keras.Model(
    inputs=[word_input, encoder_output], outputs=decoder_output
)
keras.utils.plot_model(decoder)

In [ ]:
decoder.summary()

### トレーニングモデル

エンコーダーとデコーダーを定義しました。これらを組み合わせてトレーニング用の画像モデルを作成しましょう。<br>
これには 2 つの入力（`image_input` と `word_input`）と 1 つの出力（`decoder_output`）があります。この定義はデータセットパイプラインの定義に対応している必要があります。

In [ ]:
image_caption_train_model = keras.Model(
    inputs=[image_input, word_input], outputs=decoder_output
)

### 損失関数
損失関数は単純な交差エントロピーですが、計算時にパディング（`0`）を除外する必要があります。<br>
そのため、ここでは文の長さ（0 以外の部分）を抽出し、有効な文の部分に対してのみ損失の平均を計算します。

In [ ]:
loss_object = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)


def loss_function(real, pred):
    loss_ = loss_object(real, pred)

    # 単語インデックスには 1、パディングには 0 を返す（例: [1,1,1,1,1,0,0,0,0,...,0]）
    mask = keras.ops.logical_not(keras.ops.equal(real, 0))
    mask = keras.ops.cast(mask, dtype="float32")
    loss_ *= mask

    return keras.ops.sum(loss_) / keras.ops.sum(mask)

In [ ]:
image_caption_train_model.compile(
    optimizer="adam",
    loss=loss_function,
)

## トレーニングループ

標準の `model.fit` API を使用してモデルをトレーニングできるようになりました。<br>
NVIDIA T4 GPU を使用した場合、1 エポックのトレーニングには約 15〜20 分かかります。

In [ ]:
%%time
history = image_caption_train_model.fit(batched_ds, epochs=1)

## キャプションを生成しよう！

キャプション生成中に GRU の状態を追跡し、次のタイムステップで予測された単語を入力としてデコーダーに渡す必要があるため、予測ステップはトレーニングとは異なります。

そのため、キャプション生成中に GRU の状態を保持および更新できるように、トレーニング済みの重みを使用しながら予測用の別のモデルを定義しましょう。

In [ ]:
gru_state_input = Input(shape=(ATTENTION_DIM,), name="gru_state_input")

# トレーニング済みの GRU を再利用しますが、状態（state）を受け取れるように更新します。
gru_output, gru_state = decoder_gru(embed_x, initial_state=gru_state_input)

# 他のレイヤーも同様に再利用します
context_vector = decoder_attention([gru_output, encoder_output])
addition_output = Add()([gru_output, context_vector])
layer_norm_output = layer_norm(addition_output)

decoder_output = decoder_output_dense(layer_norm_output)

# 状態の入力と出力を持つ予測モデルを定義
decoder_pred_model = keras.Model(
    inputs=[word_input, gru_state_input, encoder_output],
    outputs=[decoder_output, gru_state],
)

1. GRU の状態をゼロベクトルとして初期化します。
1. 入力画像を前処理し、エンコーダーに渡して画像の特徴量を抽出します。
1. キャプション生成を開始するために `<start>` の単語トークンを設定します。
1. for ループ内で、次の処理を行います:
    - 単語トークン（`dec_input`）、GRU 状態（`gru_state`）、および画像特徴量（`features`）を予測デコーダーに渡し、予測（`predictions`）と更新された GRU 状態を取得します。
    - ロジットから Top-K 単語を選択し、VOCAB_SIZE サイズのベクトルに対するソフトマックス計算を回避するために確率的に単語を選択します。
    - モデルが `<end>` トークンを予測した時点で予測を停止します。
    - 次のステップのために、入力単語トークンを予測された単語トークンに置き換えます。

In [ ]:
## トレーニング済みモデルを使用した確率的予測
def predict_caption(filename):
    gru_state = keras.ops.zeros((1, ATTENTION_DIM))

    img = tf.image.decode_jpeg(tf.io.read_file(filename), channels=IMG_CHANNELS)
    img = resize(img)
    img = rescale(img)

    features = encoder(keras.ops.expand_dims(img, axis=0))
    dec_input = keras.ops.expand_dims([word_to_index("<start>")], 1)
    result = []
    for i in range(MAX_CAPTION_LEN):
        predictions, gru_state = decoder_pred_model(
            [dec_input, gru_state, features]
        )

        # predictions によって与えられた対数分布からサンプリング
        top_probs, top_idxs = keras.ops.top_k(
            predictions[0][0], k=10, sorted=False
        )
        chosen_id = keras.random.categorical([top_probs], 1)[0].numpy()
        predicted_id = top_idxs.numpy()[chosen_id][0]

        result.append(tokenizer.get_vocabulary()[predicted_id])

        if predicted_id == word_to_index("<end>"):
            return img, result

        dec_input = keras.ops.expand_dims([predicted_id], 1)

    return img, result

キャプションを生成してみましょう！

In [ ]:
filename = "../sample_images/baseball.jpeg"  # surf.jpeg も試すことができます

for i in range(5):
    image, caption = predict_caption(filename)
    print(" ".join(caption[:-1]) + ".")

img = tf.image.decode_jpeg(tf.io.read_file(filename), channels=IMG_CHANNELS)
plt.imshow(img)
plt.axis("off");

いかがでしょうか？<br>
文法的に完璧ではないかもしれませんが、モデルは画像の人々、野球、ボールといった主要な要素を捉えているようです。

### オプションの課題:
独自の画像をアップロードしてキャプションを生成してみてください。<br>
また、より良いキャプションを得るためにより長くトレーニングを試すこともできます。

## まとめ
画像エンコーダーとテキストデコーダーを作成することで、画像キャプション生成モデルを構築する方法を学びました。

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.